# Financial News Sentiment Analysis for Stock Market Prediction\n\nStarter notebook to quickly experiment with the pipeline components.

In [ ]:
from src.data_collection import download_stock_data\nfrom src.preprocessing import load_sample_news, filter_news\nfrom src.sentiment_model import FinBertSentimentAnalyzer, aggregate_daily_sentiment\nfrom src.feature_engineering import compute_price_features, merge_price_and_sentiment, build_supervised_dataset\nfrom src.lstm_model import train_lstm_on_dataframe, predict_next_movement\n\nticker = "AAPL"\nstart = "2023-01-01"\nend = "2023-03-31"\n\n# 1. Stock data\nprices = download_stock_data(ticker, start, end)\n\n# 2. News + sentiment\nnews = load_sample_news()\nnews_filtered = filter_news(news, ticker=ticker, start=start, end=end)\nanalyzer = FinBertSentimentAnalyzer()\nnews_scored = analyzer.score_dataframe(news_filtered)\ndaily_sentiment = aggregate_daily_sentiment(news_scored)\n\n# 3. Features\nprice_features = compute_price_features(prices)\nmerged = merge_price_and_sentiment(price_features, daily_sentiment)\nfeature_cols = ["daily_return", "ma_close", "Volume", "daily_sentiment"]\nsupervised = build_supervised_dataset(merged, feature_columns=feature_cols)\n\n# 4. Train LSTM and predict\ntraining_result = train_lstm_on_dataframe(supervised, feature_columns=feature_cols)\npred_label, prob_up = predict_next_movement(training_result, supervised)\ndirection = "UP" if pred_label == 1 else "DOWN"\ndirection, prob_up